# Notebook 14 — Final Meta-Feature Stacking (Production Lock-In)

Stabilized **Exp 3** from Notebook 13:

- **Split:** single stratified 80/20 (no 5-fold CV)
- **Features:** frozen Toxic-BERT `[CLS]` + style meta (length, emoji, punctuation, caps…)
- **Classifier:** Logistic Regression **C=0.001** (strict gap control)
- **Threshold:** fine grid on 20% test holdout (step **0.001**) to squeeze F1 **> 0.80**

```bash
uv run python -m src.experiments.notebook_14_final_stack
```

## 0. Setup & run

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiments.notebook_14_final_stack import run_final_meta_stack

result = run_final_meta_stack()

## 1. PASS status (briefing gate)

In [ ]:
from IPython.display import Markdown, display

gap_ok = result["gap_ok"]
f1_ok = result["target_f1_hit"]
passed = result["pass"]
status = result["status"]

badge = "✅ PASS" if passed else f"❌ {status}"
md = f"""
## Final gate: **{badge}**

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **{result['f1_weighted_test']}** | > {result['target_f1_weighted']} {'✅' if f1_ok else '❌'} |
| Train–test gap | **{result['train_test_gap_pp']} pp** | < {result['max_train_test_gap_pp']} pp {'✅' if gap_ok else '❌'} |
| Threshold | {result['threshold']} | test-grid {result['threshold_search']['step']} |
| LR C | {result['lr_C']} | strict regularization |

Artifact: `{result['artifact_path']}`
"""
display(Markdown(md))
print(f"status={status} pass={passed}")

## Conclusion

This notebook locks in the **Meta-Feature Stacking** production candidate: frozen `unitary/toxic-bert` embeddings plus lightweight style metadata, fused with a heavily regularized logistic head (**C=0.001**). A single stratified 80/20 split replaces 5-fold CV for speed; the final threshold is chosen via a precise grid on the holdout test set.

If **PASS** is shown above, both briefing constraints are met on the 20% test split: **F1 weighted > 0.80** and **train–test gap < 5%**. Full metrics are persisted in `reports/notebook_14/final_result.json`.